# Build Acoustic Concept Families (ResNet293)

Creates all 3 families:
- temporal-pattern concepts
- harmonicity/noise concepts
- formant-like band dynamics concepts

Outputs: `concept/resnet_293_acoustic_concepts/concepts/*`

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/home/SpeakerRec/BioVoice') if Path('/home/SpeakerRec/BioVoice').exists() else Path.cwd().resolve()
PACK_DIR = ROOT / 'concept' / 'resnet_293_acoustic_concepts'
SRC = PACK_DIR / 'generate_all_acoustic_concepts.py'
print('ROOT=', ROOT)
print('PACK_DIR=', PACK_DIR)
print('SCRIPT exists=', SRC.exists())

In [ ]:
# Build concepts
import importlib.util
spec = importlib.util.spec_from_file_location("ac_gen", str(SRC))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

cfg = mod.GenConfig(
    seed=1337,
    samples_per_concept=200,
    nuisance_mix_prob=0.25,
    nuisance_mix_scale=0.15,
)
meta = mod.generate_all(out_root=PACK_DIR, cfg=cfg)
print(json.dumps(meta, indent=2))

In [ ]:
# Quick summary
concepts_dir = PACK_DIR / 'concepts'
dirs = sorted([d for d in concepts_dir.iterdir() if d.is_dir()])
print('num concept dirs:', len(dirs))
for d in dirs:
    n = len([p for p in d.glob('*.npy') if not p.name.startswith('_')])
    print(d.name, n)

In [ ]:
# Preview one sample from each concept
concepts_dir = PACK_DIR / 'concepts'
dirs = sorted([d for d in concepts_dir.iterdir() if d.is_dir()])
n = len(dirs)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 3.5*rows), squeeze=False)
for i, d in enumerate(dirs):
    ax = axes[i // cols][i % cols]
    first = sorted([p for p in d.glob('*.npy') if not p.name.startswith('_')])[0]
    arr = np.load(first)  # (F,T)
    im = ax.imshow(arr, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(d.name)
    ax.set_xlabel('Time')
    ax.set_ylabel('Mel')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for j in range(n, rows*cols):
    axes[j // cols][j % cols].axis("off")
plt.tight_layout()
plt.show()